# DQ-06 · promotions.csv
Checks: null rate, duplicates, domain values, temporal (start ≤ end), coverage in order_items.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
promos = pd.read_csv('promotions.csv', parse_dates=['start_date','end_date'])
print(f'Shape: {promos.shape}')
promos.head(3)

## 1. Null rate

In [ ]:
null_counts = promos.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(promos)*100).round(2)}))

## 2. Duplicate promo_id

In [ ]:
flag('Duplicate promo_id', promos.duplicated(subset='promo_id'), promos)

## 3. Domain values

In [ ]:
VALID_TYPE    = {'percentage','fixed'}
VALID_CHANNEL = {'all_channels','online','email','social_media','in_store'}

flag('Invalid promo_type',    ~promos['promo_type'].isin(VALID_TYPE),       promos)
flag('Invalid promo_channel', ~promos['promo_channel'].isin(VALID_CHANNEL), promos)
flag('discount_value ≤ 0',    promos['discount_value'] <= 0,                promos)
flag('stackable_flag not in {0,1}', ~promos['stackable_flag'].isin([0,1]),  promos)

## 4. Temporal: start_date ≤ end_date

In [ ]:
flag('start_date > end_date', promos['start_date'] > promos['end_date'], promos)
flag('end_date before 2012',  promos['end_date'] < pd.Timestamp('2012-01-01'), promos)

## 5. percentage promo: discount_value ≤ 100

In [ ]:
pct_promos = promos[promos['promo_type']=='percentage']
flag('percentage discount_value > 100', pct_promos['discount_value'] > 100, pct_promos)

## 6. Coverage: promos không được dùng trong order_items

In [ ]:
items = pd.read_csv('order_items.csv', low_memory=False)
used_promos = set(items['promo_id'].dropna())
not_used = ~promos['promo_id'].isin(used_promos)
flag('Promo defined but never used in order_items', not_used, promos)
print(promos[not_used][['promo_id','promo_name','start_date','end_date']].to_string(index=False))

## 7. Order items có promo_id nằm ngoài thời hạn promo

In [ ]:
orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
df = items.merge(orders[['order_id','order_date']], on='order_id', how='left')
df = df[df['promo_id'].notna()].merge(
    promos[['promo_id','start_date','end_date']], on='promo_id', how='left')
out_of_range = (df['order_date'] < df['start_date']) | (df['order_date'] > df['end_date'])
flag('Order date outside promo date range', out_of_range, df)

## Summary

In [ ]:
summary()